In [ ]:
import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServingEndpointDetailed, EndpointStateReady, EndpointStateConfigUpdate
import time

def get_endpoint_with_retry(w: WorkspaceClient) -> ServingEndpointDetailed:
    while True:
        try:
            endpoint = w.serving_endpoints.get("dff-orchestrator-endpoint")
            status = endpoint.state.ready

            # Check status
            if status == EndpointStateReady.READY:
                return endpoint  # Success: return the endpoint
            elif endpoint.state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
                return None     # Explicit failure: return None
            else:
                print(f"Status: pending. Retrying in 15 seconds...")
                time.sleep(15)   # Wait and retry

        except Exception as e:
            print(f"Request failed: {e}")
            return None

In [ ]:
w = WorkspaceClient()

endpoint: ServingEndpointDetailed = get_endpoint_with_retry(w)
if(endpoint):
  print(f"SUCCESS: {endpoint}")
else:
  print(f"FAILURE")